# 面试题：你会怎样设计支持短语、字段、删除和增量更新的倒排索引？

一个合格回答不能停在“用 Elasticsearch”。本 Notebook 从文档合同开始，手写 analyzer、带位置 posting、不可变 segment、tombstone、版本冲突、BM25F、短语匹配、过滤、增量刷新、compaction、相关性评估与可信快照。

受控语料只用于证明数据结构和语义。真实系统还要面对分片、副本、WAL、并发 refresh、冷热层、ACL、拼写纠错、查询规划和容量压测。

In [ ]:
import copy, hashlib, json, math, re, unicodedata, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from types import MappingProxyType
warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np

SEED67=6701
def canonical67(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def sha67(x): return hashlib.sha256(x).hexdigest()
assert np.__version__ and SEED67==6701

## 1. 先定义文档、版本与 analyzer 合同

文档包含稳定 `doc_id`、单调递增 `version`、`tenant`、`title/body` 和可过滤 `category`。同一 ID 的旧版本必须拒绝；删除写 tombstone，不能立即修改旧 segment。analyzer 做 Unicode NFKC、英文小写和中英文词元切分，并同时保留位置，短语查询才有可能实现。

analyzer 版本是索引协议的一部分。若写入端和查询端 normalization 不同，系统不会报错，只会悄悄漏召回。

In [ ]:
@dataclass(frozen=True)
class Document67:
    doc_id: str; version: int; tenant: str; title: str; body: str; category: str

class Analyzer67:
    version="nfkc-lower-zh-char-enword-v1"
    def analyze(self,text):
        if not isinstance(text,str): raise TypeError("text_must_be_str")
        normalized=unicodedata.normalize("NFKC",text).lower()
        return re.findall(r"[a-z0-9]+|[\u4e00-\u9fff]",normalized)

analyzer67=Analyzer67()
assert analyzer67.analyze("ＡI搜索 Search-2")==["ai","搜","索","search","2"]
assert analyzer67.analyze("")==[] and analyzer67.version.endswith("v1")
try: analyzer67.analyze(None); raise AssertionError("non-string accepted")
except TypeError as e: assert str(e)=="text_must_be_str"
docs67=[
    Document67("d1",1,"acme","混合搜索","关键词 检索 与 向量 召回", "search"),
    Document67("d2",1,"acme","关键词检索","倒排 索引 支持 精确 短语", "search"),
    Document67("d3",1,"acme","向量检索","近似 最近邻 向量 索引", "vector"),
    Document67("d4",1,"beta","搜索安全","关键词 检索 需要 租户 权限", "security"),
    Document67("d5",1,"acme","排序评估","召回 后 使用 重排 模型", "ranking"),
]
assert len({d.doc_id for d in docs67})==len(docs67) and all(d.version>0 for d in docs67)

## 2. 不可变 segment 与带位置 posting

posting 不是只有 doc ID，还要保存字段词频与位置。每个 refresh 构建一个不可变 segment：`term -> field -> doc_id -> positions`，另存字段长度和原始文档。不可变结构让读请求无需持有大锁；更新通过新 segment 覆盖旧版本，后台再 compact。

这里用深拷贝和只读访问模拟不可变边界。生产中通常把词典、posting block、skip data 和 stored fields 编码成不同文件，并做 checksum。

In [ ]:
class Segment67:
    def __init__(self,documents,analyzer,generation):
        self.generation=int(generation); self.analyzer_version=analyzer.version
        self.documents={d.doc_id:d for d in documents}; self.postings=defaultdict(lambda:defaultdict(dict)); self.lengths={}
        for d in documents:
            for field in ("title","body"):
                tokens=analyzer.analyze(getattr(d,field)); self.lengths[(d.doc_id,field)]=len(tokens)
                positions=defaultdict(list)
                for pos,term in enumerate(tokens): positions[term].append(pos)
                for term,pos_list in positions.items(): self.postings[term][field][d.doc_id]=tuple(pos_list)
    def docs_for(self,term,field): return self.postings.get(term,{}).get(field,{})

seg67=Segment67(docs67,analyzer67,1)
assert seg67.docs_for("检","body")["d1"]==(3,) and seg67.docs_for("检","title")["d2"]==(3,)
assert seg67.lengths[("d1","title")]==4 and seg67.documents["d3"].category=="vector"
assert seg67.docs_for("不存在","body")=={}

## 3. 增量写入、删除与读时版本解析

Index 维护 segment 列表、最新版本表和 tombstone。写入先检查版本，再生成新 segment；删除也要求版本递增。查询遍历 segment 时，只接受“该 posting 所属文档版本等于最新版本且未删除”的记录，避免旧 segment 复活。

真实服务需要 WAL 与原子提交点，保证崩溃后 segment 文件、版本表和 tombstone 一致。本例显式暴露 `refresh`，便于理解 near-real-time 可见性。

In [ ]:
class InvertedIndex67:
    def __init__(self,analyzer): self.analyzer=analyzer; self.segments=[]; self.latest={}; self.tombstones={}; self.pending=[]; self.generation=0
    def upsert(self,doc):
        if not isinstance(doc,Document67): raise TypeError("document_contract")
        if doc.version<=self.latest.get(doc.doc_id,0): raise ValueError("stale_version")
        self.latest[doc.doc_id]=doc.version; self.tombstones.pop(doc.doc_id,None); self.pending.append(doc)
    def delete(self,doc_id,version):
        if version<=self.latest.get(doc_id,0): raise ValueError("stale_version")
        self.latest[doc_id]=int(version); self.tombstones[doc_id]=int(version)
    def refresh(self):
        if self.pending:
            self.generation+=1; self.segments.append(Segment67(self.pending,self.analyzer,self.generation)); self.pending=[]
        return self.generation
    def live_documents(self):
        out={}
        for seg in self.segments:
            for doc_id,d in seg.documents.items():
                if self.latest.get(doc_id)==d.version and doc_id not in self.tombstones: out[doc_id]=d
        return out

index67=InvertedIndex67(analyzer67)
for d in docs67: index67.upsert(d)
assert index67.live_documents()=={} and index67.refresh()==1 and len(index67.live_documents())==5
updated67=Document67("d2",2,"acme","关键词短语检索","倒排 索引 支持 关键词 精确 短语", "search")
index67.upsert(updated67); index67.refresh()
assert index67.live_documents()["d2"].version==2 and len(index67.segments)==2
try: index67.upsert(docs67[1]); raise AssertionError("stale update accepted")
except ValueError as e: assert str(e)=="stale_version"

## 4. 词项、短语、字段和过滤查询

查询先解析双引号短语，其余词元作为 SHOULD terms。短语命中要求同一字段内连续位置相差 1；不能把 title 末尾与 body 开头拼成短语。tenant 与 category 在候选阶段过滤，权限必须先于打分和日志输出。

面试中要说明查询语言边界：是否支持 AND/OR/NOT、转义、最大子句数、空查询与超长 token。这里实现一个最小但严格的协议。

In [ ]:
@dataclass(frozen=True)
class Query67:
    terms: tuple; phrases: tuple; tenant: str; category: str|None=None

def parse_query67(text,tenant,category=None):
    if not text.strip() or not tenant: raise ValueError("query_contract")
    raw_phrases=re.findall(r'"([^"]+)"',text); remainder=re.sub(r'"[^"]+"',' ',text)
    phrases=tuple(tuple(analyzer67.analyze(p)) for p in raw_phrases)
    if any(not p for p in phrases): raise ValueError("empty_phrase")
    return Query67(tuple(analyzer67.analyze(remainder)),phrases,tenant,category)

def phrase_match67(segment,doc_id,field,phrase):
    if not phrase: return False
    position_sets=[set(segment.docs_for(term,field).get(doc_id,())) for term in phrase]
    return any(all(start+i in position_sets[i] for i in range(len(phrase))) for start in position_sets[0])

q67=parse_query67('关键词 "精确 短语"',"acme","search")
assert q67.terms==("关","键","词") and q67.phrases==(("精","确","短","语"),)
assert phrase_match67(index67.segments[-1],"d2","body",q67.phrases[0])
assert not phrase_match67(index67.segments[0],"d1","body",q67.phrases[0])
try: parse_query67("   ","acme"); raise AssertionError("empty query accepted")
except ValueError as e: assert str(e)=="query_contract"

## 5. 手写 BM25F 与候选生成

BM25 的 IDF 使用 live corpus 计算；字段分别归一化长度，再乘字段权重。这里将同一 term 在 title/body 的饱和贡献相加，短语命中另加固定 bonus。候选是任一 term/phrase token 的 posting 并集，然后执行 tenant/category 和 phrase MUST 过滤。

IDF、平均长度和字段权重都属于索引/模型版本。小语料会出现极端 IDF，因此线上还需要 minimum-should-match、停用词和 query-length normalization。

In [ ]:
class BM25F67:
    def __init__(self,k1=1.2,b=.75,weights=None): self.k1=k1; self.b=b; self.weights=weights or {"title":2.,"body":1.}
    def score(self,index,query):
        live=index.live_documents(); allowed={i:d for i,d in live.items() if d.tenant==query.tenant and (query.category is None or d.category==query.category)}
        all_terms=set(query.terms)
        for p in query.phrases: all_terms.update(p)
        avg={f:np.mean([sum(index.analyzer.analyze(getattr(d,f)).count(t) for t in [])+len(index.analyzer.analyze(getattr(d,f))) for d in live.values()]) for f in ("title","body")}
        scores={doc_id:0. for doc_id in allowed}
        for term in all_terms:
            containing=set()
            for seg in index.segments:
                for field in self.weights: containing.update(i for i in seg.docs_for(term,field) if i in live and live[i].version==seg.documents[i].version)
            idf=math.log(1+(len(live)-len(containing)+.5)/(len(containing)+.5))
            for doc_id,d in allowed.items():
                for field,w in self.weights.items():
                    toks=index.analyzer.analyze(getattr(d,field)); tf=toks.count(term)
                    if tf: scores[doc_id]+=idf*w*(tf*(self.k1+1))/(tf+self.k1*(1-self.b+self.b*len(toks)/avg[field]))
        for doc_id,d in list(allowed.items()):
            for phrase in query.phrases:
                matched=False
                for seg in index.segments:
                    if doc_id in seg.documents and seg.documents[doc_id].version==d.version:
                        matched=any(phrase_match67(seg,doc_id,f,phrase) for f in self.weights)
                if not matched: scores.pop(doc_id,None); break
                scores[doc_id]+=2.5
        return sorted(scores.items(),key=lambda x:(-x[1],x[0]))

scorer67=BM25F67(); results67=scorer67.score(index67,q67)
assert results67 and results67[0][0]=="d2" and all(index67.live_documents()[d].tenant=="acme" for d,_ in results67)
assert all(index67.live_documents()[d].category=="search" for d,_ in results67)
assert scorer67.score(index67,parse_query67("向量","acme"))[0][0]=="d3"

## 6. 删除、compaction 与快照隔离

删除只追加 tombstone，旧 segment 仍占空间。compaction 读取一个一致性快照中的 live documents，重写单一 segment，再安全回收旧文件。查询请求应固定 generation，不能在遍历一半时看到 refresh 后的新 segment。

本例用显式 copy 模拟 snapshot；生产中常用 commit point 引用一组不可变 segment，并通过引用计数延迟删除。

In [ ]:
snapshot_before67=copy.deepcopy(index67.live_documents())
index67.delete("d3",2)
assert "d3" not in index67.live_documents() and "d3" in snapshot_before67
post_delete67=scorer67.score(index67,parse_query67("向量","acme"))
assert "d3" not in {doc_id for doc_id,_ in post_delete67} and post_delete67[0][0]=="d1"
try: index67.delete("d3",1); raise AssertionError("stale delete accepted")
except ValueError as e: assert str(e)=="stale_version"

def compact67(index):
    live=list(index.live_documents().values()); index.generation+=1
    index.segments=[Segment67(live,index.analyzer,index.generation)] if live else []
    return len(live)
old_segments67=len(index67.segments); live_count67=compact67(index67)
assert old_segments67==2 and len(index67.segments)==1 and live_count67==4
assert set(index67.segments[0].documents)==set(index67.live_documents())

## 7. 相关性评估不能只看“搜到了”

构造 query-level gold，手写 DCG/nDCG、Recall@k 和零相关查询策略。评估集要按查询族、时间和租户切分；同义改写若跨 split，会让离线数字虚高。除相关性外还要单独统计权限泄漏、空结果率、P50/P99 latency 和索引新鲜度。

小 gold 的 nDCG 只验证排序公式，不能证明字段权重可迁移到真实业务。

In [ ]:
def dcg67(grades): return sum((2**g-1)/math.log2(i+2) for i,g in enumerate(grades))
def ndcg67(ranked,gold,k):
    actual=[gold.get(d,0) for d in ranked[:k]]; ideal=sorted(gold.values(),reverse=True)[:k]; denom=dcg67(ideal)
    return dcg67(actual)/denom if denom else 0.
gold67={"d2":3,"d1":1}; ranked67=[d for d,_ in scorer67.score(index67,parse_query67("关键词","acme"))]
metric67=ndcg67(ranked67,gold67,3)
assert 0<=metric67<=1 and ranked67[0]=="d2"
assert ndcg67(["d2","d1"],gold67,2)==1.
assert ndcg67([],{},3)==0. and dcg67([0,0])==0.
assert ndcg67(["d1","d2"],gold67,2)<1.

## 8. 发布快照与面试总结

manifest 绑定 analyzer、字段、权重、segment generation、live document 摘要和 tombstone 语义。服务加载时必须从实际文档重算摘要，不信任包内自报值。真实系统还应绑定 posting 编码版本、分片路由、ACL schema 和 commit checksum。

面试回答顺序可以是：目标/SLO → 文档与 analyzer 合同 → segment/posting → 查询规划/BM25F → 增量和删除 → 评估 → 分片与故障。这样比“上 ES”更能展示设计能力。

In [ ]:
def corpus_digest67(index):
    rows=[d.__dict__ for d in sorted(index.live_documents().values(),key=lambda x:x.doc_id)]
    return sha67(canonical67(rows).encode())
manifest67={"artifact_id":"inverted-index-demo-v1","analyzer":analyzer67.version,"fields":{"title":2.,"body":1.},"generation":index67.generation,"corpus_digest":corpus_digest67(index67),"delete":"versioned_tombstone","phrase":"same_field_consecutive_positions"}
TRUST67=MappingProxyType({manifest67["artifact_id"]:sha67(canonical67(manifest67).encode())})
def load_snapshot67(index,manifest):
    actual=copy.deepcopy(manifest); actual["corpus_digest"]=corpus_digest67(index)
    digest=sha67(canonical67(actual).encode())
    if TRUST67.get(actual.get("artifact_id"))!=digest: raise RuntimeError("untrusted_index_snapshot")
    if actual["analyzer"]!=index.analyzer.version or actual["generation"]!=index.generation: raise RuntimeError("index_protocol_mismatch")
    return MappingProxyType(copy.deepcopy(actual))
published67=load_snapshot67(index67,manifest67)
assert published67["corpus_digest"]==corpus_digest67(index67) and isinstance(TRUST67,MappingProxyType)
tampered67=copy.deepcopy(manifest67); tampered67["fields"]["title"]=99.
try: load_snapshot67(index67,tampered67); raise AssertionError("tampered manifest accepted")
except RuntimeError as e: assert str(e)=="untrusted_index_snapshot"
print({"live_docs":live_count67,"segments":len(index67.segments),"ndcg":round(metric67,4)})

## 9. 复杂度、失败模式与来源

写入成本与 token 数线性相关；term 查询近似与 posting 长度相关；短语查询还要做位置交集。常见错误包括 analyzer 漂移、旧版本复活、先打分后 ACL、跨字段短语、IDF 混用删除文档、refresh 非原子和 compaction 提前回收。

- Manning et al., [Introduction to Information Retrieval: Index construction](https://nlp.stanford.edu/IR-book/html/htmledition/index-construction-1.html)。
- Robertson & Zaragoza, [The Probabilistic Relevance Framework: BM25](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf)。
- Apache Lucene, [Index File Formats](https://lucene.apache.org/core/9_9_0/core/org/apache/lucene/codecs/lucene99/package-summary.html)。